# Random Forest Regression

training random forest regression in blocks using baseline features + traficom features

## 1. Imports

This notebook mirrors the linear-regression workflow but swaps the estimator for `RandomForestRegressor`. It now adds a compact tuning block across a few stronger random-forest configurations, target modes, and listing-date variants before reusing the selected setup across the feature-set comparison.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 2. Load data

In [2]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [3]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

parsed_first_seen_series = [
    pd.to_datetime(dataset_df["first_seen_date"], errors="coerce")
    for dataset_df in [train_df, validation_df]
    if "first_seen_date" in dataset_df.columns
]

if len(parsed_first_seen_series) > 0:
    reference_first_seen_date = min(series.min() for series in parsed_first_seen_series)

    for dataset_df in [train_df, validation_df]:
        first_seen_date = pd.to_datetime(dataset_df["first_seen_date"], errors="coerce")
        last_seen_date = pd.to_datetime(dataset_df["last_seen_date"], errors="coerce")

        dataset_df["first_seen_day_offset"] = (
            first_seen_date - reference_first_seen_date
        ).dt.days
        dataset_df["last_seen_day_offset"] = (
            last_seen_date - reference_first_seen_date
        ).dt.days
        dataset_df["listing_midpoint_day_offset"] = (
            dataset_df["first_seen_day_offset"]
            + dataset_df["last_seen_day_offset"]
        ) / 2

## 3. Quick data checks

In [4]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 79)
Validation shape: (1689, 79)


In [5]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 79 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   product_id                      7954 non-null   int64  
 1   part_name                       7954 non-null   object 
 2   price                           7954 non-null   float64
 3   quality_grade                   7954 non-null   object 
 4   oem_number                      7954 non-null   object 
 5   mileage                         7954 non-null   float64
 6   brand                           7954 non-null   object 
 7   model                           7954 non-null   object 
 8   category                        7954 non-null   object 
 9   subcategory                     7954 non-null   object 
 10  scrape_date                     7954 non-null   object 
 11  year_start                      7954 non-null   int64  
 12  year_end               

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [6]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column in train_df.columns
]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
]

random_forest_date_offset_features = [
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_date": "Observed listing start within the crawl window; often informative for the tree models.",
    "last_seen_date": "Observed listing end within the crawl window; often informative for the tree models.",
    "first_seen_day_offset": "Numeric listing-start offset used in the compact random-forest tuning block.",
    "last_seen_day_offset": "Numeric listing-end offset used in the compact random-forest tuning block.",
    "listing_midpoint_day_offset": "Numeric midpoint of the listing visibility window for random forest.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

recommended_excluded_features = set(recommended_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price"
    and column not in recommended_excluded_features
    and column not in random_forest_date_offset_features
]

recommended_model_features_without_listing_dates = [
    column for column in recommended_model_features
    if column not in {"first_seen_date", "last_seen_date"}
]

recommended_model_features_with_offsets = list(dict.fromkeys(
    recommended_model_features_without_listing_dates
    + random_forest_date_offset_features
))

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nNumber of recommended model features:", len(recommended_model_features))
print("Number of recommended model features with date offsets:", len(recommended_model_features_with_offsets))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,first_seen_date,missing_from_previous_manual_all,Observed listing start within the crawl window...
3,last_seen_date,missing_from_previous_manual_all,Observed listing end within the crawl window; ...
4,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
5,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
6,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
7,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
8,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
9,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...


## 5. Select baseline feature set

In [7]:
features_baseline_only = baseline_features

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 13


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status']

## 6. Build X and y

In [8]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [9]:
y_train_log = np.log(y_train)
y_validation_log = np.log(y_validation)

y_train_log.head()

0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [10]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage', 'year_start', 'year_end', 'year_span', 'year_mid']

## 8. Detect column types

In [11]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [12]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,5,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,8,"[part_name, quality_grade, oem_number, brand, ..."


## 9. Tune Random Forest Pipeline

In [13]:
def build_random_forest_pipeline(
    X_train_current,
    model_params,
    onehot_min_frequency=5,
):
    numeric_features_current = X_train_current.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()
    categorical_features_current = X_train_current.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline_current = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=onehot_min_frequency)),
    ])

    preprocessor_current = ColumnTransformer(transformers=[
        ("num", numeric_pipeline_current, numeric_features_current),
        ("cat", categorical_pipeline_current, categorical_features_current),
    ])

    final_model_params = {
        "random_state": 42,
        "n_jobs": -1,
    }
    final_model_params.update(model_params)

    return Pipeline(steps=[
        ("preprocessor", preprocessor_current),
        ("model", RandomForestRegressor(**final_model_params)),
    ])


def prepare_random_forest_target(y_series, target_mode):
    if target_mode == "log":
        return np.log(y_series)
    if target_mode == "raw":
        return y_series.copy()

    raise ValueError(f"Unsupported target_mode: {target_mode}")


def convert_random_forest_predictions_to_eur(predictions, target_mode):
    if target_mode == "log":
        return np.exp(predictions)
    if target_mode == "raw":
        return predictions

    raise ValueError(f"Unsupported target_mode: {target_mode}")


random_forest_candidate_configs = {
    "log_reference": {
        "target_mode": "log",
        "onehot_min_frequency": 5,
        "model_params": {
            "n_estimators": 300,
            "min_samples_leaf": 2,
        },
        "notes": "Original notebook configuration for comparison.",
    },
    "log_half_features_leaf_1": {
        "target_mode": "log",
        "onehot_min_frequency": 5,
        "model_params": {
            "n_estimators": 400,
            "min_samples_leaf": 1,
            "max_features": 0.5,
        },
        "notes": "More flexible trees with a lower feature-subsampling fraction.",
    },
    "log_half_features_leaf_2": {
        "target_mode": "log",
        "onehot_min_frequency": 5,
        "model_params": {
            "n_estimators": 500,
            "min_samples_leaf": 2,
            "max_features": 0.5,
        },
        "notes": "Slightly more conservative version of the half-feature setup.",
    },
    "raw_half_features_leaf_1": {
        "target_mode": "raw",
        "onehot_min_frequency": 5,
        "model_params": {
            "n_estimators": 400,
            "min_samples_leaf": 1,
            "max_features": 0.5,
        },
        "notes": "Direct price prediction to test whether MAE improves without the log target.",
    },
    "raw_half_features_leaf_2": {
        "target_mode": "raw",
        "onehot_min_frequency": 5,
        "model_params": {
            "n_estimators": 500,
            "min_samples_leaf": 2,
            "max_features": 0.5,
        },
        "notes": "Direct price prediction with slightly stronger leaf regularization.",
    },
}

random_forest_tuning_feature_sets = {
    "recommended_raw_dates": recommended_model_features,
    "recommended_date_offsets": recommended_model_features_with_offsets,
}

tuning_results = []
for feature_variant_name, tuning_features in random_forest_tuning_feature_sets.items():
    X_train_tuning = train_df[tuning_features].copy()
    X_validation_tuning = validation_df[tuning_features].copy()

    for config_name, config in random_forest_candidate_configs.items():
        model_current = build_random_forest_pipeline(
            X_train_tuning,
            model_params=config["model_params"],
            onehot_min_frequency=config["onehot_min_frequency"],
        )
        model_current.fit(
            X_train_tuning,
            prepare_random_forest_target(y_train, config["target_mode"]),
        )

        validation_pred_model_scale_current = model_current.predict(X_validation_tuning)
        validation_pred_current = convert_random_forest_predictions_to_eur(
            validation_pred_model_scale_current,
            config["target_mode"],
        )

        tuning_results.append({
            "config": config_name,
            "feature_variant": feature_variant_name,
            "target_mode": config["target_mode"],
            "onehot_min_frequency": config["onehot_min_frequency"],
            "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
            "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
            "validation_R2": r2_score(y_validation, validation_pred_current),
            "notes": config["notes"],
        })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_MAE", "validation_RMSE"]
).reset_index(drop=True)

selected_random_forest_config_name = tuning_results_df.iloc[0]["config"]
selected_random_forest_feature_variant_name = tuning_results_df.iloc[0]["feature_variant"]
selected_random_forest_config = random_forest_candidate_configs[selected_random_forest_config_name]

print("Selected random forest config:", selected_random_forest_config_name)
print("Selected tuning feature variant:", selected_random_forest_feature_variant_name)
print(selected_random_forest_config)

display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)

In [14]:
baseline_model = build_random_forest_pipeline(
    X_train,
    model_params=selected_random_forest_config["model_params"],
    onehot_min_frequency=selected_random_forest_config["onehot_min_frequency"],
)

selected_random_forest_config

## 10. Train Tuned Baseline Random Forest

In [15]:
baseline_model.fit(
    X_train,
    prepare_random_forest_target(y_train, selected_random_forest_config["target_mode"]),
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 11. Predict and convert back to euro scale

In [16]:
validation_pred_model_scale = baseline_model.predict(X_validation)

In [17]:
validation_pred = convert_random_forest_predictions_to_eur(
    validation_pred_model_scale,
    selected_random_forest_config["target_mode"],
)

## 12. Preview Baseline Random Forest Predictions


In [18]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,176.847865
1,473.6,395.955250
2,142.1,144.535280
3,82.9,84.066524
4,177.6,112.616540


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [19]:
features_baseline_plus_traficom = baseline_features + traficom_features

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 26


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [20]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [21]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [22]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Build A Fresh Tuned Random Forest Pipeline

In [23]:
print("Using the selected random forest config for baseline + Traficom:")
print(selected_random_forest_config_name)
selected_random_forest_config

In [24]:
random_forest_traficom = build_random_forest_pipeline(
    X_train_traficom,
    model_params=selected_random_forest_config["model_params"],
    onehot_min_frequency=selected_random_forest_config["onehot_min_frequency"],
)

## 17. Train baseline + Traficom random forest

In [25]:
random_forest_traficom.fit(
    X_train_traficom,
    prepare_random_forest_target(y_train, selected_random_forest_config["target_mode"]),
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 18. Predict on validation and convert back to euro scale

In [26]:
validation_pred_model_scale_traficom = random_forest_traficom.predict(X_validation_traficom)

In [27]:
validation_pred_traficom = convert_random_forest_predictions_to_eur(
    validation_pred_model_scale_traficom,
    selected_random_forest_config["target_mode"],
)

## 19. Preview Baseline + Traficom Random Forest Predictions

In [28]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,176.849078
1,473.6,397.090464
2,142.1,144.138174
3,82.9,84.571467
4,177.6,112.480825


In [29]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,252.287219
std,567.153248,575.324597
min,5.900000,6.933116
25%,59.000000,57.676441
50%,100.600000,91.196193
75%,236.000000,225.262697
max,4284.000000,4258.152689


## 20. All Recommended Features

This section trains the random forest model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.


In [30]:
features_all = recommended_model_features

print("Number of recommended model features:", len(features_all))
features_all

Number of recommended model features: 72


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [31]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [32]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [33]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Build A Fresh Tuned Random Forest Pipeline

In [34]:
print("Using the selected random forest config for the recommended-feature model:")
print(selected_random_forest_config_name)
selected_random_forest_config

In [35]:
random_forest_all = build_random_forest_pipeline(
    X_train_all,
    model_params=selected_random_forest_config["model_params"],
    onehot_min_frequency=selected_random_forest_config["onehot_min_frequency"],
)

## 24. Train baseline + Traficom + registry lifecycle + listing dynamics random forest

In [36]:
random_forest_all.fit(
    X_train_all,
    prepare_random_forest_target(y_train, selected_random_forest_config["target_mode"]),
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 25. Predict on validation and convert back to euro scale

In [37]:
validation_pred_model_scale_all = random_forest_all.predict(X_validation_all)

In [38]:
validation_pred_all = convert_random_forest_predictions_to_eur(
    validation_pred_model_scale_all,
    selected_random_forest_config["target_mode"],
)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [39]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,177.553946
1,473.6,540.476965
2,142.1,142.067312
3,82.9,82.868426
4,177.6,177.538070


In [40]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,247.852754
std,567.153248,531.345448
min,5.900000,5.901338
25%,59.000000,59.173398
50%,100.600000,100.652702
75%,236.000000,236.708379
max,4284.000000,4245.887353


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing random forest model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [41]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",177.553946,0.046054,0.002121
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",540.476965,66.876965,4472.528457
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",142.067312,0.032688,0.001068
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",82.868426,0.031574,0.000997
4,177.6,airbag,all,"Side airbags - , e-(Right)",177.538070,0.061930,0.003835


In [42]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [43]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,5.10,0.35,16.22,90.1%,91.7%,94.5%
1,vehicle exterior / suspension,294,6.20,0.34,21.37,90.1%,92.9%,96.9%
2,electric / transmitter / databox / sensor,404,7.04,0.29,33.61,87.1%,94.1%,97.5%
3,gear box / drive axle / middle axle,241,10.98,0.97,37.54,72.2%,86.3%,95.4%
4,brakes,214,12.99,0.84,57.14,89.7%,92.5%,96.7%
5,airbag,106,16.61,0.60,50.31,84.0%,84.9%,87.7%
6,engine,249,70.20,0.44,389.42,77.1%,85.5%,94.0%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,70.20,0.44,389.42,77.1%,85.5%,94.0%
5,airbag,106,16.61,0.60,50.31,84.0%,84.9%,87.7%
4,brakes,214,12.99,0.84,57.14,89.7%,92.5%,96.7%
3,gear box / drive axle / middle axle,241,10.98,0.97,37.54,72.2%,86.3%,95.4%
2,electric / transmitter / databox / sensor,404,7.04,0.29,33.61,87.1%,94.1%,97.5%
1,vehicle exterior / suspension,294,6.20,0.34,21.37,90.1%,92.9%,96.9%
0,fuel,181,5.10,0.35,16.22,90.1%,91.7%,94.5%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,440.40,38.11,1073.34,9.5%,28.6%,57.1%
22,passenger airbag,25,50.26,2.36,94.60,56.0%,56.0%,64.0%
21,deliverer,23,16.84,0.71,34.36,78.3%,78.3%,78.3%
20,fuse box / electricity central,27,11.94,5.96,24.50,70.4%,92.6%,92.6%
19,airbag control unit,20,11.25,1.08,27.43,85.0%,90.0%,90.0%
18,abs hydraulic pump,36,8.72,2.01,16.86,83.3%,83.3%,100.0%
17,adaptiv farthållare sensor,24,6.45,6.03,8.45,58.3%,100.0%,100.0%
16,gear box bracket,31,5.92,0.18,15.06,77.4%,96.8%,96.8%
15,left,33,4.75,0.27,10.30,78.8%,97.0%,100.0%
14,all,153,4.65,0.47,16.73,92.8%,95.4%,98.7%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,1.14,0.35,2.87,100.0%,100.0%,100.0%
1,Drive shaft -(Left front),32,2.14,0.81,5.44,93.8%,96.9%,100.0%
2,Brake Caliper -(Left front),20,3.58,0.53,12.83,95.0%,95.0%,95.0%
3,Wheel bearing spindle shaft -(Left rear),21,4.66,0.73,17.15,95.2%,95.2%,95.2%
4,Trailing link rear -(Left),20,5.62,0.42,9.58,70.0%,100.0%,100.0%
5,Wheel bearing spindle shaft -(Right rear),20,5.63,0.63,15.58,90.0%,90.0%,100.0%
6,Drive shaft -(Right front),23,10.55,1.35,22.09,78.3%,87.0%,95.7%
7,ABS Hydraulic pump -,24,11.07,1.99,20.34,75.0%,75.0%,100.0%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,ABS Hydraulic pump -,24,11.07,1.99,20.34,75.0%,75.0%,100.0%
6,Drive shaft -(Right front),23,10.55,1.35,22.09,78.3%,87.0%,95.7%
5,Wheel bearing spindle shaft -(Right rear),20,5.63,0.63,15.58,90.0%,90.0%,100.0%
4,Trailing link rear -(Left),20,5.62,0.42,9.58,70.0%,100.0%,100.0%
3,Wheel bearing spindle shaft -(Left rear),21,4.66,0.73,17.15,95.2%,95.2%,95.2%
2,Brake Caliper -(Left front),20,3.58,0.53,12.83,95.0%,95.0%,95.0%
1,Drive shaft -(Left front),32,2.14,0.81,5.44,93.8%,96.9%,100.0%
0,Distributors Vacuum regulator -,20,1.14,0.35,2.87,100.0%,100.0%,100.0%


## 29. Select The Best Random Forest Feature Set

Reuse the tuned random-forest configuration selected earlier across every feature-set experiment so the comparison stays focused on feature value instead of estimator noise.

In [44]:
excluded_features = recommended_excluded_features

feature_sets = {
    "baseline only": baseline_features,
    "baseline + traficom_core": baseline_features + traficom_features,
    "baseline + traficom_core + registry_lifecycle": (
        baseline_features + traficom_features + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        baseline_features
        + traficom_features
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "previous manual all-features set": manual_all_feature_groups,
    "all recommended features": recommended_model_features,
    "all recommended features without listing dates": recommended_model_features_without_listing_dates,
    "all recommended features with date offsets": recommended_model_features_with_offsets,
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [45]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )

In [46]:
for model_name, feature_list in feature_sets.items():
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    model_current = build_random_forest_pipeline(
        X_train_current,
        model_params=selected_random_forest_config["model_params"],
        onehot_min_frequency=selected_random_forest_config["onehot_min_frequency"],
    )
    model_current.fit(
        X_train_current,
        prepare_random_forest_target(y_train, selected_random_forest_config["target_mode"]),
    )

    validation_pred_model_scale_current = model_current.predict(X_validation_current)
    validation_pred_current = convert_random_forest_predictions_to_eur(
        validation_pred_model_scale_current,
        selected_random_forest_config["target_mode"],
    )
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })

## 30. Validate The Best Random Forest Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [47]:
# Rebuild the best-model validation dataframe if this section is run on its own.
if "best_model_validation_df" not in globals():
    if "best_experiment_name" not in globals():
        if "experiment_results" not in globals() or len(experiment_results) == 0:
            raise NameError(
                "best_experiment_name is not defined. Run the feature-set comparison section first."
            )

        experiment_results_df = pd.DataFrame(experiment_results)
        best_experiment_name = experiment_results_df.sort_values(
            ["validation_MAE", "validation_RMSE"]
        ).iloc[0]["experiment"]

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,205,35.4,40.8,5.36
1,"(5.899, 47.4]",underpredicted,144,35.6,34.8,0.81
2,"(47.4, 82.6]",overpredicted,166,62.9,65.1,2.23
3,"(47.4, 82.6]",underpredicted,169,59.4,57.1,2.28
4,"(82.6, 154.06]",overpredicted,152,111.9,114.4,2.52
5,"(82.6, 154.06]",underpredicted,177,108.2,106.0,2.18
6,"(154.06, 248.42]",overpredicted,149,204.0,206.9,2.93
7,"(154.06, 248.42]",underpredicted,189,206.0,198.2,7.77
8,"(248.42, 4284.0]",overpredicted,143,805.1,830.2,25.16
9,"(248.42, 4284.0]",underpredicted,195,941.6,828.8,112.88


In [48]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,3.48,9.78,11.0%,-0.05
1,"(47.4, 82.6]",335,47.6,82.6,61.1,2.26,6.69,3.7%,+0.01
2,"(82.6, 154.06]",329,82.8,153.9,109.9,2.34,8.92,2.3%,+0.03
3,"(154.06, 248.42]",338,154.1,248.3,205.1,5.64,17.52,2.7%,+0.09
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,75.77,341.97,6.7%,+0.55


In [49]:
experiment_results_df = pd.DataFrame(experiment_results)
baseline_row = experiment_results_df.loc[
    experiment_results_df["experiment"] == "baseline only"
].iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

display_columns = [
    "experiment",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

best_experiment_name = experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]


experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})

,experiment,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
5,all recommended features,72,17.9132,-6.2714,1,153.3244,+83.0164,5,0.9269
4,previous manual all-features set,68,18.0172,-6.1674,2,153.3547,+83.0467,6,0.9268
2,baseline + traficom_core + registry_lifecycle,40,23.8357,-0.3489,3,69.3195,-0.9885,1,0.9851
1,baseline + traficom_core,26,23.8579,-0.3267,4,70.4433,+0.1353,3,0.9846
3,baseline + traficom_core + registry_lifecycle + traficom_extended,62,23.9555,-0.2291,5,70.6335,+0.3255,4,0.9845
0,baseline only,13,24.1846,+0.0000,6,70.3080,+0.0000,2,0.9846
